## Read the tissue positions and metadata file
This gives us information on the format of pixel value coordinates.
It also gives us an info on outliers and e.g. how many negative examples entries in the y-position-column there are. 

In [8]:
tissuepositions

,barcode,in_tissue,array_row,array_col,pxl_row_in_fullres,pxl_col_in_fullres
0,s_008um_00000_00000-1,0,0,0,36591.694839,-242.091470
1,s_008um_00000_00001-1,0,0,1,36591.965228,-212.877209
2,s_008um_00000_00002-1,0,0,2,36592.235617,-183.662943
3,s_008um_00000_00003-1,0,0,3,36592.506006,-154.448672
4,s_008um_00000_00004-1,0,0,4,36592.776395,-125.234396
...,...,...,...,...,...,...
702239,s_008um_00837_00833-1,1,837,833,12357.868203,24323.438541
702240,s_008um_00837_00834-1,1,837,834,12358.136674,24352.657678
702241,s_008um_00837_00835-1,1,837,835,12358.405145,24381.876820
702242,s_008um_00837_00836-1,1,837,836,12358.673616,24411.095967


In [13]:
print("positive pixel values: ", len(tissuepositions[tissuepositions["pxl_col_in_fullres"] > 0]))
print("negative pixel values: ", len(tissuepositions[tissuepositions["pxl_col_in_fullres"] < 0]))
print("in_tissue == 1: ", len(tissuepositions[tissuepositions["in_tissue"] == 1]))
print("in_tissue == 0: ", len(tissuepositions[tissuepositions["in_tissue"] == 0]))

positive pixel values:  698148
negative pixel values:  4096
in_tissue == 1:  507684
in_tissue == 0:  194560


In [20]:
col = "pxl_col_in_fullres"
smallest_gt_zero = tissuepositions.loc[tissuepositions[col] > 0, col].min()
print(f"Smallest positive value in {col}: {smallest_gt_zero}")

smallest_gt_zero = tissuepositions.loc[tissuepositions[col] > 0, col].max()
print(f"Biggest positive value in {col}: {smallest_gt_zero}")

Smallest positive value in pxl_col_in_fullres: 0.01616907989636651
Biggest positive value in pxl_col_in_fullres: 24440.315118149927


## IDK what this does

In [4]:
# Count occurrences of each id
id_count_check = len(results["id"].unique())
id_count = results["id"].value_counts()
print("Unique IDs found in the results:", id_count_check)  # List of unique IDs
print("Counts of each ID:", id_count)  # Counts of each ID

# Filter brands that occur more than once
ids_more_than_once = id_count[id_count > 1]

# If you want just the list of brand names:
id_label = ids_more_than_once.index.tolist()

# If you want the number of such brands:
number_of_ids = len(ids_more_than_once)

print("IDs appearing more than once:")
print(ids_more_than_once)       # brand name and counts
#print(id_label)                 # just brand names
print("Number of Ids: ", number_of_ids)           # count of brands appearing more than once


Unique IDs found in the results: 110016
Counts of each ID: id
ID_85299     7
ID_534315    5
ID_381216    5
ID_84659     5
ID_276597    5
            ..
ID_391996    1
ID_390550    1
ID_391503    1
ID_390132    1
ID_388651    1
Name: count, Length: 110016, dtype: int64
IDs appearing more than once:
id
ID_85299     7
ID_534315    5
ID_381216    5
ID_84659     5
ID_276597    5
            ..
ID_54843     2
ID_535021    2
ID_531992    2
ID_401223    2
ID_540878    2
Name: count, Length: 3687, dtype: int64
Number of Ids:  3687


In [1]:
import pandas as pd
df1 = pd.read_csv("uni/notebooks/metadata/WSI_patch_metadata1.csv")
df2 = pd.read_csv("uni/notebooks/metadata/WSI_patch_metadata2.csv")

merged_df = pd.concat([df1, df2], axis=0, ignore_index=True)

merged_df.to_csv("uni/notebooks/metadata/WSI_patch_metadata.csv", index=False)

In [2]:
import scanpy as sc
import anndata as ad

# Load both .h5ad files
adata1 = sc.read_h5ad("uni/notebooks/metadata/WSI_patch_embeddings1.h5ad")
adata2 = sc.read_h5ad("uni/notebooks/metadata/WSI_patch_embeddings2.h5ad")

# Concatenate along observations (rows)
adata_merged = ad.concat([adata1, adata2], axis=0, join="outer")  
# join="outer" keeps all obs/var, join="inner" keeps only common

# Save
adata_merged.write_h5ad("uni/notebooks/metadata/WSI_patch_embeddings.h5ad")


## Checking coordinates of filtered patches and Nuclei

In [1]:
import os
import scanpy as sc
import pandas as pd
import geopandas as gpd


person = "CRC_P1"
bin_size = "002um"

uni_folder = "uni/notebooks/metadata"
uni_patch_embeddings_filename = "filtered_WSI_patch_embeddings.h5ad"

cell_seg_folder = f"Cell_Segmentation/output/{person}/{bin_size}/custom_binning"
filtered_bins_filename = "filtered_custom_bins.h5ad"
nuclei_genes_filtered_filename = "count_area_filtered_adata.h5ad"
nuclei_polys_filename = "nuclei_geometry.parquet"

In [2]:
# Loading the data
adata_patches = sc.read_h5ad(os.path.join(uni_folder, uni_patch_embeddings_filename))
#adata_bins = sc.read_h5ad(os.path.join(cell_seg_folder, filtered_bins_filename))
nuclei_positions = gpd.read_parquet(os.path.join(cell_seg_folder, nuclei_polys_filename))
adata_filtered_nuclei = sc.read_h5ad(os.path.join(cell_seg_folder, nuclei_genes_filtered_filename))

# merge nuclei positions with adata_filtered_nuclei
adata_filtered_nuclei.obs = adata_filtered_nuclei.obs.merge(nuclei_positions, on="id", how="left")


/home/charlottewolf/miniconda3/envs/NucleiSeg/lib/python3.11/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [ ]:
print("Min x:", adata_patches.obs.loc[adata_patches.obs["Slide_ID"]==person, "X"].min())
print("Max x:", adata_patches.obs.loc[adata_patches.obs["Slide_ID"]==person, "X"].max())
print("Min y:", adata_patches.obs.loc[adata_patches.obs["Slide_ID"]==person, "Y"].min())
print("Max y:", adata_patches.obs.loc[adata_patches.obs["Slide_ID"]==person, "Y"].max())

Min x: 448
Max x: 24192


In [10]:
# filter rows/columns > 0
row_vals = tissuepositions.loc[tissuepositions['pxl_row_in_fullres'] > 0, 'pxl_row_in_fullres']
col_vals = tissuepositions.loc[tissuepositions['pxl_col_in_fullres'] > 0, 'pxl_col_in_fullres']

# compute min and max
min_row, max_row = row_vals.min(), row_vals.max()
min_col, max_col = col_vals.min(), col_vals.max()

print(f"Row -> min > 0: {min_row}, max: {max_row}")
print(f"Column -> min > 0: {min_col}, max: {max_col}")

Row -> min > 0: 12137.797400349906, max: 36828.94892524755
Column -> min > 0: 0.0003797818750692132, max: 24436.630813568463


In [8]:
import scanpy as sc
import pandas as pd
import os

h5ad_test = sc.read_h5ad("Cell_Segmentation/output/merged_custom_binning/filtered_custom_bins.h5ad")

In [7]:
import scanpy as sc
import pandas as pd
import geopandas as gpd

person = "CRC_P1"
bin_size = "002um"

adata_filtered_nuclei = sc.read_h5ad("Cell_Segmentation/output/merged_custom_binning/002um/count_area_filtered_adata.h5ad")
nuclei_positions = gpd.read_parquet("Cell_Segmentation/output/merged_custom_binning/002um/nuclei_geometry.parquet")

#adata_filtered_nuclei.obs = adata_filtered_nuclei.obs.merge(nuclei_positions, on=["id", "Slide_ID"], how="left")
adata_filtered_nuclei.obs = adata_filtered_nuclei.obs.merge(nuclei_positions, on=["Slide_ID", "id"], how="left")
adata_filtered_nuclei = adata_filtered_nuclei[adata_filtered_nuclei.obs["Slide_ID"] == person]


# gives us the overview over each nucleus for each person, with its geometry
# adata_filtered_nuclei.obs

In [10]:
nuclei_positions = nuclei_positions[nuclei_positions["Slide_ID"] == person]
# nuclei_positions

# Get the bounding box of the whole dataset
minx, miny, maxx, maxy = nuclei_positions.total_bounds

print("Bounds from Nuclei Positiosn Parquet file:")
print("Min X:", minx)
print("Min Y:", miny)
print("Max X:", maxx)
print("Max Y:", maxy)


minx2, miny2, maxx2, maxy2 = adata_gdf.total_bounds

print("Bounds from merged adata file:")
print("Min X:", minx2)
print("Min Y:", miny2)
print("Max X:", maxx2)
print("Max Y:", maxy2)


Bounds from Nuclei Positiosn Parquet file:
Min X: 680.6541748046875
Min Y: 14102.367919921875
Max X: 24408.420166015625
Max Y: 36828.15768432617
Bounds from merged adata file:
Min X: 680.6541748046875
Min Y: 14102.367919921875
Max X: 24408.420166015625
Max Y: 36828.15768432617
